# Using Tools
This notebook demonstrates how we can use tools to gather information and provide more accurate responses. We'll build a weather-checking mock as an example.

## What we'll learn:
- Basic interaction with Language Models (LLM)
- How to create and use tools with AI
- The complete flow of an AI agent using tools
- Understanding the message flow in a tool-enabled conversation

### Setup

In [1]:
import json
from dotenv import load_dotenv
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper
import os

In [2]:
load_dotenv()
assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("OPENAI_BASE_URL") is not None
assert os.getenv("TAVILY_API_KEY") is not None

In [3]:
chat_model = LLM()

## Basic LLM Interaction
Before we dive into tools, let's understand how to interact with our Language Model in its simplest form. 
There are two main ways to communicate with the model:

In [4]:
# Method 1: Simple single-turn query
response = chat_model.invoke("What is an AI Agent?")
print("Single Query Response:\n", response.content)

Single Query Response:
 An **AI agent** is a system that can **perceive information, make decisions, and take actions** to achieve a goal, often with some level of autonomy.

### Simple definition
Think of it as software that doesn’t just answer questions — it can also **do things**.

### Core parts of an AI agent
- **Input / perception:** It receives data from the environment  
  - e.g. text, images, sensor data, web pages, API responses
- **Decision-making:** It chooses what to do next based on its goal
- **Action:** It performs an action  
  - e.g. send a message, call an API, search the web, control a robot
- **Feedback loop:** It observes the result and adjusts its next step

### Examples
- A chatbot that answers questions
- A customer support agent that checks a database and issues refunds
- A robot vacuum that maps a room and cleans it
- A trading bot that monitors markets and places orders

### In modern AI
People often use “AI agent” to mean an LLM-based system that can:
- pla

In [5]:
# Method 2: Multi-turn conversation with specific roles
# Pydantic モデルを渡すと、OpenAIのなかで自動的にJSONスキーマに変換して処理してくれる
messages = [
    SystemMessage(content="You're an OpenAI API specialist"),
    UserMessage(content="What is Function Calling?")
]
response = chat_model.invoke(messages)
print("\nStructured Conversation Response:\n", response.content)


Structured Conversation Response:
 **Function Calling** is a feature that lets an AI model **request that your application run a specific function** with structured arguments, instead of just replying in plain text.

### In simple terms
You give the model a list of functions it can use, such as:

- `get_weather(city)`
- `search_products(query)`
- `create_calendar_event(date, title)`

If the user asks, “What’s the weather in Paris?”, the model can respond with something like:

```json
{
  "name": "get_weather",
  "arguments": {
    "city": "Paris"
  }
}
```

Your app then executes that function, gets the result, and sends it back to the model if needed.

### Why it’s useful
Function calling helps the model:

- **Use real data** from APIs or databases
- **Take actions** in your app
- **Produce structured output** reliably
- **Reduce hallucinations** by grounding responses in external systems

### Common use cases
- Weather lookup
- Booking appointments
- Database queries
- Sending email

## Building an AI Tool
Now let's make our AI more capable by giving it a tool to check the weather. 
This demonstrates how we can extend AI capabilities beyond just conversation.

### Understanding the Tool Structure:
1. We use the `@tool` decorator to mark a function as an AI tool
2. The tool needs clear documentation and typed parameters
3. The tool should return structured data

In [6]:
@tool
def get_weather(city: str):
    """Get the current temperature for a city.

    Args:
        city (str): Name of the city to check weather for

    Returns:
        dict: Contains temperature information for the requested city
    """
    # In a real application, this would call a weather API
    mock_weather = {
        "São Paulo": "28°C",
        "Oslo": "-3°C",
        "New York": "15°C",
        "Tokyo": "22°C"
    }
    # dictのgetで値が見つからないときは"Unknown"を返す
    return {"temperature": mock_weather.get(city, "Unknown")}

In [7]:
# Bind the tool to an LLM
chat_model_with_tools = LLM(tools=[get_weather])

## Understanding the Tool Usage Flow
Let's break down how the AI uses tools step by step:

1. User asks a question about weather
2. AI recognizes the need to use the weather tool
3. AI makes a tool call
4. Tool executes and returns results
5. AI processes the tool's response
6. AI formulates a natural language response

Let's see this in action:

In [8]:
# Set up our system with clear instructions
messages = [
    SystemMessage(
        content="You are a helpful assistant that can access a tool to get current temperature "
                "for cities. Use the tool whenever someone asks about the weather or temperature "
                "in a specific location. Inform the user if you don't know the answer."
    ),
    UserMessage(content="How cold is it in Oslo?")
]

In [9]:
# AI decides to use the weather tool
ai_message = chat_model_with_tools.invoke(messages)
ai_message

AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BHyqKUSCXN3vfq44dWXBzgkr', function=Function(arguments='{"city":"Oslo"}', name='get_weather'), type='function')])

In [10]:
# Check messages structure
messages.append(ai_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get current temperature for cities. Use the tool whenever someone asks about the weather or temperature in a specific location. Inform the user if you don't know the answer.", role='system'),
 UserMessage(content='How cold is it in Oslo?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BHyqKUSCXN3vfq44dWXBzgkr', function=Function(arguments='{"city":"Oslo"}', name='get_weather'), type='function')])]

In [11]:
# Tool call id will be required later when creating the ToolMessage
tool_call_id = messages[-1].tool_calls[0].id
tool_call_id

'call_BHyqKUSCXN3vfq44dWXBzgkr'

In [12]:
# Extract the arguments
args = json.loads(messages[-1].tool_calls[0].function.arguments)
args

{'city': 'Oslo'}

In [13]:
# Execute the tool with the AI's requested parameters
tool_result = get_weather(**args)
tool_result

{'temperature': '-3°C'}

In [14]:
# Create a tool response message
tool_message = ToolMessage(
    content=tool_result["temperature"],
    tool_call_id=tool_call_id,
    name="get_weather"
)
tool_message

ToolMessage(content='-3°C', role='tool', tool_call_id='call_BHyqKUSCXN3vfq44dWXBzgkr', name='get_weather')

In [15]:
# Check messages structure
messages.append(tool_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get current temperature for cities. Use the tool whenever someone asks about the weather or temperature in a specific location. Inform the user if you don't know the answer.", role='system'),
 UserMessage(content='How cold is it in Oslo?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BHyqKUSCXN3vfq44dWXBzgkr', function=Function(arguments='{"city":"Oslo"}', name='get_weather'), type='function')]),
 ToolMessage(content='-3°C', role='tool', tool_call_id='call_BHyqKUSCXN3vfq44dWXBzgkr', name='get_weather')]

In [16]:
# Let AI formulate final response
ai_message = chat_model_with_tools.invoke(messages)
ai_message

AIMessage(content='It’s currently **-3°C in Oslo**.', role='assistant', tool_calls=None)

In [17]:
# Check messages structure
messages.append(ai_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get current temperature for cities. Use the tool whenever someone asks about the weather or temperature in a specific location. Inform the user if you don't know the answer.", role='system'),
 UserMessage(content='How cold is it in Oslo?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BHyqKUSCXN3vfq44dWXBzgkr', function=Function(arguments='{"city":"Oslo"}', name='get_weather'), type='function')]),
 ToolMessage(content='-3°C', role='tool', tool_call_id='call_BHyqKUSCXN3vfq44dWXBzgkr', name='get_weather'),
 AIMessage(content='It’s currently **-3°C in Oslo**.', role='assistant', tool_calls=None)]